<a href="https://colab.research.google.com/github/vamsikadaru/The-Day-Trip-Genie/blob/main/The_Day_Trip_Genie.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Part 0: Setup & Authentication 🔑

First things first, let's get all our tools ready. This step installs the necessary libraries and securely configures your Google API key so your agents can access the power of Gemini.

In [ ]:
!pip install google-adk google-generativeai -q

# --- Import all necessary libraries ---
import os
import sys
import json
import asyncio
import random
import string
from uuid import uuid4
from typing import Any, List

import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, Markdown, display

# --- ADK, Agent, and Evaluation Components ---
from google.adk.agents import Agent
from google.adk.events import Event
from google.adk.runners import Runner
import google.adk as adk
from google.adk.tools import google_search
from google.adk.sessions import InMemorySessionService, Session
from google.genai import types
from google.genai.types import Content, Part


print("✅ All libraries are ready to go!")



✅ All libraries are ready to go!


### Configure Your API Key
To use Gemini models, you need an API key from [Google AI Studio](https://aistudio.google.com/app/apikey). This section securely collects your key and configures it for the ADK.


In [ ]:
# --- API Key Configuration ---
from google.colab import userdata

# Option 1: Use Colab Secrets (recommended)
# Go to the 🔑 icon in the left sidebar, add a secret named GOOGLE_API_KEY
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    print("✅ API key loaded from Colab Secrets.")
except Exception:
    # Option 2: Paste it directly (less secure but fine for learning)
    import getpass
    GOOGLE_API_KEY = getpass.getpass("🔑 Enter your Google AI Studio API key: ")
    print("✅ API key entered manually.")


✅ API key loaded from Colab Secrets.


In [ ]:
# --- Set Environment Variables for ADK ---

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"

print(f"✅ API key configured (starts with '{GOOGLE_API_KEY[:6]}...')")
print("✅ Using Google AI Studio (not Vertex AI).")


✅ API key configured (starts with 'AQ.Ab8...')
✅ Using Google AI Studio (not Vertex AI).


```
+--------------------------------------------------+
|         Spontaneous Day Trip Agent 🤖            |
|--------------------------------------------------|
|  Model: gemini-2.5-flash                         |
|  Description:                                    |
|   Generates full-day trip itineraries based on   |
|   mood, interests, and budget                    |
|--------------------------------------------------|
|  🔧 Tools:                                       |
|   - Google Search                                |
|--------------------------------------------------|
|  🧠 Capabilities:                                |
|   - Budget Awareness (cheap / splurge)           |
|   - Mood Matching (adventurous, relaxing, etc.)  |
|   - Real-Time Info (hours, events)               |
|   - Morning / Afternoon / Evening plan           |
+--------------------------------------------------+

            ▲
            |
    +------------------+
    |   User Input     |
    |------------------|
    |  Mood            |
    |  Interests       |
    |  Budget          |
    +------------------+

            |
            ▼

+--------------------------------------------------+
|             Output: Markdown Itinerary           |
|--------------------------------------------------|
| - Time blocks (Morning / Afternoon / Evening)    |
| - Venue names with links and hours               |
| - Budget-matching activities                     |
+--------------------------------------------------+
```


In [ ]:
# --- Agent Definition ---

def create_day_trip_agent():
    """Create the Spontaneous Day Trip Generator agent"""
    return Agent(
        name="day_trip_agent",
        model="gemini-2.5-flash",
        description="Agent specialized in generating spontaneous full-day itineraries based on mood, interests, and budget.",
        instruction="""
        You are the "Spontaneous Day Trip" Generator 🚗 - a specialized AI assistant that creates engaging full-day itineraries.

        Your Mission:
        Transform a simple mood or interest into a complete day-trip adventure with real-time details, while respecting a budget.

        Guidelines:
        1. **Budget-Aware**: Pay close attention to budget hints like 'cheap', 'affordable', or 'splurge'. Use Google Search to find activities (free museums, parks, paid attractions) that match the user's budget.
        2. **Full-Day Structure**: Create morning, afternoon, and evening activities.
        3. **Real-Time Focus**: Search for current operating hours and special events.
        4. **Mood Matching**: Align suggestions with the requested mood (adventurous, relaxing, artsy, etc.).

        RETURN itinerary in MARKDOWN FORMAT with clear time blocks and specific venue names.
        """,
        tools=[google_search]
    )

day_trip_agent = create_day_trip_agent()
print(f"🧞 Agent '{day_trip_agent.name}' is created and ready for adventure!")

🧞 Agent 'day_trip_agent' is created and ready for adventure!


In [ ]:
# --- A Helper Function to Run Our Agents ---
# We'll use this function throughout the notebook to make running queries easy.

async def run_agent_query(agent: Agent, query: str, session: Session, user_id: str, is_router: bool = False):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n🚀 Running query for agent: '{agent.name}' in session: '{session.id}'...")

    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )

    final_response = ""
    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user")
        ):
            if not is_router:
                # Let's see what the agent is thinking!
                print(f"EVENT: {event}")
            if event.is_final_response():
                final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"An error occurred: {e}"

    if not is_router:
     print("\n" + "-"*50)
     print("✅ Final Response:")
     display(Markdown(final_response))
     print("-"*50 + "\n")

    return final_response

# --- Initialize our Session Service ---
# This one service will manage all the different sessions in our notebook.
session_service = InMemorySessionService()
my_user_id = "adk_adventurer_001"

In [ ]:
# --- Let's test the Day Trip Genie! ---

async def run_day_trip_genie():
    # Create a new, single-use session for this query
    day_trip_session = await session_service.create_session(
        app_name=day_trip_agent.name,
        user_id=my_user_id
    )

    # Note the new budget constraint in the query!
    query = "Plan a relaxing and artsy day trip near Sunnyvale, CA. Keep it affordable!"
    print(f"🗣️ User Query: '{query}'")

    await run_agent_query(day_trip_agent, query, day_trip_session, my_user_id)

await run_day_trip_genie()

🗣️ User Query: 'Plan a relaxing and artsy day trip near Sunnyvale, CA. Keep it affordable!'

🚀 Running query for agent: 'day_trip_agent' in session: '01b3975e-e8d8-47a3-a7e2-116186e8e30a'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Here's a relaxing and artsy day trip itinerary near Sunnyvale, California, designed to be affordable for Monday, June 15, 2026:

## Relaxing & Artsy Day Trip: Stanford & Palo Alto

This itinerary focuses on the beautiful Stanford University campus and charming downtown Palo Alto, offering free art experiences and serene natural spaces.

**Budget Focus:** This trip leverages free admission at world-class museums and gardens, with costs primarily for parking and meals. Parking at Stanford is paid on weekdays until 4 PM via the ParkMobile app, and free on weekends.

---

### **Morning (9:30 AM - 12:00 PM): Serene Stroll through the Arizona Garden**

Start your day with a peaceful visit to the **Stanford Universi

Here's a relaxing and artsy day trip itinerary near Sunnyvale, California, designed to be affordable for Monday, June 15, 2026:

## Relaxing & Artsy Day Trip: Stanford & Palo Alto

This itinerary focuses on the beautiful Stanford University campus and charming downtown Palo Alto, offering free art experiences and serene natural spaces.

**Budget Focus:** This trip leverages free admission at world-class museums and gardens, with costs primarily for parking and meals. Parking at Stanford is paid on weekdays until 4 PM via the ParkMobile app, and free on weekends.

---

### **Morning (9:30 AM - 12:00 PM): Serene Stroll through the Arizona Garden**

Start your day with a peaceful visit to the **Stanford University Arizona Garden**. This historic cactus and succulent garden offers a unique botanical experience, perfect for a relaxing walk. Designed in the late 19th century, it features a diverse collection of desert plants from both the Eastern and Western Hemispheres. It's open from sunrise to sunset and admission is free.

*   **Location:** South side of the Mausoleum off Quarry Road, between Campus Drive and Arboretum Road, Stanford, CA.
*   **Getting There:** From Sunnyvale, it's a short drive to the Stanford campus. Park in the visitor parking spaces near the Anderson Collection and Cantor Arts Center, which is about a 5-10 minute walk to the garden. Remember to use the ParkMobile app for payment if visiting before 4 PM on a weekday.
*   **Cost:** Free admission.

### **Lunch (12:00 PM - 1:30 PM): Affordable Bites in Palo Alto**

Head to downtown Palo Alto for an affordable and delicious lunch. You'll find a variety of options catering to different tastes.

*   **Suggestions:**
    *   **Oren's Hummus:** Known for fresh ingredients, generous portions, and reasonable prices, offering a delightful array of hummus plates, shawarma wraps, and falafel bowls.
    *   **Zareen's:** Offers flavorful and budget-friendly Pakistani and Indian cuisine, including biryanis and wraps. Don't miss their chai tea!
    *   **Coconuts Caribbean Restaurant & Bar:** A family-owned spot bringing vibrant Caribbean flavors like jerk chicken and fried plantains to Palo Alto in a laid-back atmosphere.
    *   **Crepevine Restaurants:** Offers a wide variety of sweet and savory crepes, omelets, and salads at reasonable prices.
*   **Location:** Downtown Palo Alto, easily accessible from Stanford University.
*   **Cost:** ~$10-$20 per person.

### **Afternoon (1:30 PM - 4:30 PM): Modern Art at the Anderson Collection**

Immerse yourself in modern and contemporary American art at the **Anderson Collection at Stanford University**. This museum houses an extraordinary collection of paintings and sculptures, including works from the New York School and Bay Area Figuration. Admission is always free, and advance reservations are not required.

*   **Location:** 375 Lomita Dr, Stanford, CA. It's conveniently located on the Stanford campus, a short walk from where you might have parked for the Arizona Garden.
*   **Operating Hours (Monday):** 11:00 AM - 5:00 PM.
*   **Cost:** Free admission.

### **Evening (4:30 PM onwards): Downtown Palo Alto Stroll & Casual Dinner**

After your dose of art, head back to downtown Palo Alto. Enjoy a relaxing stroll along University Avenue and its surrounding streets, browsing local shops, or simply people-watching.

*   **Activity:** Explore the vibrant streets of downtown Palo Alto.
*   **Dinner:** For an affordable dinner, consider returning to one of the lunch spots you might have missed, or try another budget-friendly option like:
    *   **Ramen Nagi:** A popular Japanese ramen spot if you're in the mood for a hearty bowl.
    *   **Mediterranean Wraps:** Known for tasty and filling wraps and friendly service.
*   **Cost:** Variable, depending on your dinner choice.

This itinerary offers a perfect blend of nature's tranquility and artistic inspiration, all while keeping your budget in mind. Enjoy your spontaneous day trip!

--------------------------------------------------



---
## Part 2: Supercharging Agents with Custom Tools 🛠️

So far, we've used the powerful built-in `GoogleSearch` tool. But the true power of agents comes from connecting them to your own logic and data sources.

This is where **custom tools** come in. Let's explore three patterns for giving your agent new skills, using real-world, practical examples.

### 2.1 The Simple `FunctionTool`: Calling a Real-Time Weather API

The most direct way to create a tool is by writing a Python function. This is perfect for synchronous tasks like fetching data from an API.

**Key Concept:** The function's **docstring** is critical. The ADK uses it as the tool's official description, which the LLM reads to understand its purpose, parameters, and when to use it.

In this example, we'll create a tool that calls the **free, public U.S. National Weather Service API** to get a real-time forecast. No API key needed!

In [ ]:
# --- Tool Definition: A function that calls a live public API ---
import requests
import json

# A simple lookup to avoid needing a separate geocoding API for this example
LOCATION_COORDINATES = {
    "sunnyvale": "37.3688,-122.0363",
    "san francisco": "37.7749,-122.4194",
    "lake tahoe": "39.0968,-120.0324"
}

def get_live_weather_forecast(location: str) -> dict:
    """Gets the current, real-time weather forecast for a specified location in the US.

    Args:
        location: The city name, e.g., "San Francisco".

    Returns:
        A dictionary containing the temperature and a detailed forecast.
    """
    print(f"🛠️ TOOL CALLED: get_live_weather_forecast(location='{location}')")

    # Find coordinates for the location
    normalized_location = location.lower()
    coords_str = None
    for key, val in LOCATION_COORDINATES.items():
        if key in normalized_location:
            coords_str = val
            break
    if not coords_str:
        return {"status": "error", "message": f"I don't have coordinates for {location}."}

    try:
        # NWS API requires 2 steps: 1. Get the forecast URL from the coordinates.
        points_url = f"https://api.weather.gov/points/{coords_str}"
        headers = {"User-Agent": "ADK Example Notebook"}
        points_response = requests.get(points_url, headers=headers)
        points_response.raise_for_status() # Raise an exception for bad status codes
        forecast_url = points_response.json()['properties']['forecast']

        # 2. Get the actual forecast from the URL.
        forecast_response = requests.get(forecast_url, headers=headers)
        forecast_response.raise_for_status()

        # Extract the relevant forecast details
        current_period = forecast_response.json()['properties']['periods'][0]
        return {
            "status": "success",
            "temperature": f"{current_period['temperature']}°{current_period['temperatureUnit']}",
            "forecast": current_period['detailedForecast']
        }
    except requests.exceptions.RequestException as e:
        return {"status": "error", "message": f"API request failed: {e}"}

# --- Agent Definition: An agent that USES the new tool ---

weather_agent = Agent(
    name="weather_aware_planner",
    model="gemini-2.5-flash",
    description="A trip planner that checks the real-time weather before making suggestions.",
    instruction="You are a cautious trip planner. Before suggesting any outdoor activities, you MUST use the `get_live_weather_forecast` tool to check conditions. Incorporate the live weather details into your recommendation.",
    tools=[get_live_weather_forecast]
)

print(f"🌦️ Agent '{weather_agent.name}' is created and can now call a live weather API!")

In [ ]:
# --- Let's test the Weather-Aware Planner ---

async def run_weather_planner_test():
    weather_session = await session_service.create_session(app_name=weather_agent.name, user_id=my_user_id)
    query = "I want to go hiking near Lake Tahoe, what's the weather like?"
    print(f"🗣️ User Query: '{query}'")
    await run_agent_query(weather_agent, query, weather_session, my_user_id)

await run_weather_planner_test()